## DSPy Quantized Qwen2 Information Extraction Simplified with Mlflow Tracking

#### Load in Python Libraries

In [1]:
import os 
import sys
import re
from dotenv import load_dotenv
load_dotenv()
pythonpath = os.getenv('PYTHONPATH')
if pythonpath:
    sys.path.extend(pythonpath.split(os.pathsep))

import dspy
from rich import print
import pandas as pd
import torch
import gc

from dspy.teleprompt import BootstrapFewShot

from dspy.evaluate.evaluate import Evaluate

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2','rougeL'], use_stemmer=True)
import mlflow 

#### Set mlFlow Experiment

In [2]:
mlflow.set_experiment("adeptID")
mlflow.start_run(run_name = "DSPy-QA-pipeline")

2024/07/01 21:47:45 INFO mlflow.tracking.fluent: Experiment with name 'adeptID' does not exist. Creating a new experiment.


<ActiveRun: >

#### Helper Functions

In [3]:
def validate_ans(example, pred, trace = None):
    gold = example.info.lower()
    print(gold)
    prediction = pred.info.split('Answer: ')[1].lower()
    print(prediction)

    scores = scorer.score(gold, prediction)
    score2 = scores['rouge2'][0]
    score1 = scores['rouge1'][0]
    scoreL = scores['rougeL'][0]
    score = (0.2*score1 + 0.3*score2 + 0.5* scoreL)

    print(score)

    return score

#### Load in Test Examples

In [4]:
test_examples = pd.read_csv(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\data\Manual Labeling - Sheet2.csv", header = None)
test_examples_list = [
    '{"position_title": "Senior Inside Sales Rep/Sales Engineer", "location": "Walpole, MA", "work_arrangement": "Hybrid", "experience": "Depends on Experience", "employment_type": "Full Time", "pay": "$120K/year", "degree": "A BS in the Engineering field", "certification": "CRM (salesforce.com), RFQ experience and price quotes to the DOD", "required_skills": "Inside/Outside Technical Sales Experience, Experience working with Outside Sales Reps"}',
    '{"position_title": "Penetration Tester", "location": "Washington, DC", "work_arrangement": "On-site", "experience": "10+ years of Penetration Testing experience", "employment_type": "Full time", "pay": "Not specified", "degree": "Bachelors Degree in Computer Science", "certification": "Offensive Security certification (OSCP, OSCE), GIAC certification (GPEN, GWAPT, GXPN), or technology specific certification (MCSE, LPIC, CCNA)", "required_skills": "NIST guidance, FedRAMP control baseline, industry best practice"}',
    '{"position_title": "NURSES - RNs or LPNs", "location": "Sudbury, MA 01776", "work_arrangement": "on-site", "experience": "Minimum of 1 year Long term care experience/SNF experience preferred", "employment_type": "full-time", "pay": "HOURLY - EVERY OTHER WEEKEND REQUIRED", "degree": "Must have a valid MA Nursing License", "certification": "RN or LPN License in Massachusetts", "required_skills": "Medication pass, treatments, resident care"}',
    '{"position_title": "Planner IV - Transportation Planner", "location": "Yakima, WA, 98901", "work_arrangement": "On-site", "experience": "5 years of increasingly responsible professional experience", "employment_type": "Full-Time", "pay": "$39.84 - $50.53 Hourly", "degree": "Bachelor\'s Degree in Planning or other related field", "certification": "None specified", "required_skills": "Transportation planning, coordination with the Yakama Nation, preparation of loans and grants"}',
    '{"position_title": "Associate Attorney", "location": "McAllen, TX", "work_arrangement": "on-site", "experience": "None specified.", "employment_type": "full-time", "pay": "$50,000 a year", "degree": "Law doctoral degree", "certification": "Admission to the state bar and in good standing with the relevant jurisdiction.", "required_skills": "Interest in Family and Criminal Law, proven track record of successful hearing coverage and strong advocacy skills."}',
    '{"position_title": "EMT-Advanced-Emergency Medical Service", "location": "Rosenberg, TX 77471", "work_arrangement": "On-site", "experience": "Pre-hospital experience preferred, experience in a high performance ALS system", "employment_type": "Full time", "pay": "$2,008.28 - $2,421.41 biweekly", "degree": "High school diploma/GED", "certification": "paramedic certification or EMS degree, AEMT, enrolled in an EMT Paramedic Program, DSHS EMT-Advanced, valid Texas driver\'s license", "required_skills": "Strong verbal and written communication, organizational skills, interpersonal skills, judgment, reasoning, decision-making, teaching"}',
    '{"position_title": "Transportation Environmental Resources Specialist", "location": "Weston, West Virginia 26452-8289", "work_arrangement": "On-site", "experience": "24 Months", "employment_type": "Full time Permanent", "pay": "$1,700.00 - $2,521.15 Biweekly", "degree": "Bachelor\'s degree from a regionally accredited college or university with a major in archeology, chemistry, geology, history, physics, geography, biology, economics, engineering, environmental studies, natural science, or a related field.", "certification": "Drivers license, DL", "required_skills": "Full-performance level, complex professional work in a specialty area in the acquisition, preservation, management and protection of the state\'s environmental/natural resources."}',
    '{"position_title": "Hotel Front Desk Clerk", "location": "La Quinta Inn & Suites, USF Tampa, FL", "work_arrangement": "On-site", "experience": "At least one year of hospitality industry experience", "employment_type": "Full Time", "pay": "$14 hourly", "degree": "High school diploma or GED", "certification": "None specified", "required_skills": "Customer service, Microsoft Office, organizational skills, communication, time management"}',
    '{"position_title": "Psychotherapist", "location": "Asbury, NJ", "work_arrangement": "On-site", "experience": "1 year", "employment_type": "Hourly", "pay": "$65 - $95 an hour", "degree": "Doctor of Psychology Doctoral degree or equivalent", "certification": "LSW Social Work License, LCSW, LPC, LAC, or other relevant licenses", "required_skills": "experience with children, strong interpersonal skills, ability to establish rapport with clients"}',
    '{"position_title": "Cryptocurrency / FX Trader - Entry Level", "location": "Not specified", "work_arrangement": "Remote", "experience": "No prior experience required", "employment_type": "Full-time or part-time", "pay": "Results-based commissions and performance bonuses", "degree": "Bachelor\'s degree in finance, economics, or related field preferred", "certification": "None specified", "required_skills": "Strong analytical skills, quick decision-making"}'
]

#### Set LLM Qwen2

In [5]:
llm = dspy.HFModel(model="unsloth/Qwen2-7B-bnb-4bit", hf_device_map='auto', model_kwargs= {'temperature':0.0,'do_sample': True, })
dspy.settings.configure(lm = llm)

c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


#### Create DSPy Signature and Module

In [6]:
class GenerateAnswer(dspy.Signature):
    """Extract information from a job posting and return the output in a json format.
    If you don't know the answer, return 'Not Specified'. Should be key-value with output as a dictionary."""
    context = dspy.InputField(desc="contain relevant facts")
    question = dspy.InputField(desc="unique possible questions")
    answer = dspy.OutputField(desc="key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words")

class QuestionAnswer(dspy.Module):
    def __init__(self, question):
        super().__init__()
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer, max_tokens=400)
        self.question = question
    def forward(self, context):
        context = self._preprocess_context(context)
        prediction = self.generate_answer(context=context, question=self.question)
        self._cleanup_memory()
        return dspy.Prediction(context=context, info=self._postprocess_prediction(prediction))
    
    @staticmethod
    def normalize(job_post: str) -> str:
        job_post = job_post.strip('\n')
        job_post = re.sub(r'^[^\w\s]+|[^\w\s]+$', '', job_post, flags=re.UNICODE)
        job_post = job_post.strip('\n')
        return job_post.strip().lower()
    
    @staticmethod
    def _preprocess_context(context):
        context = context.replace('\n', ' ').replace('“', '"').replace('”', '"')
        context = QuestionAnswer.normalize(context)
        context = re.sub(r'[^\w\s]', '', context)  # Remove punctuation, keep words and numbers
        return context
    
    @staticmethod
    def _postprocess_prediction(prediction):
        return re.sub(r"```\n|```", "", prediction.answer)
    
    @staticmethod
    def _cleanup_memory():
        torch.cuda.empty_cache()
        gc.collect()

info_extract = QuestionAnswer('''
    "position_title": What is the title of this position?
    "location": Where is this position located, including city, state, and zip code?
    "work_arrangement": What is the work arrangement for this position, remote, hybrid, or on-site?
    "experience": What are the years of experience required for this position?
    "employment_type": What is the employment type, full time, part time, or internship?
    "pay": What is the pay for this position?
    "degree": What is the required degree?
    "certifications": What certifications or qualifications are required?
    "required_skills": What are the required skills?
''')

In [7]:
print(test_examples[1][2])

NURSES - RNS & LPNs IN SNF - SIGN ON BONUS - CHILD DAYCARE ON SITE (Sudbury) Sudbury Pines Extended Care Facility 
Inc. compensation: HOURLY - EVERY OTHER WEEKEND REQUIRED employment type: full-time job title: NURSE URSES - RNs or
LPNs Sudbury Pines Extended Care Sudbury, MA 01776 Sudbury Pines Extended Care Facility is seeking Nurses to join 
our team! Currently seeking the following positions: Full time or Part Time RN or LPN Sudbury Pines Extended Care 
facility is a 92 bed facility located in the MetroWest area. We are a single family owned facility who strives to 
provide quality care in a home-like and family oriented environment for our residents. Responsibilities: 
Responsible for the overall nursing care and delivery of resident services - medication pass, treatments, resident 
quality of life, etc. Manage staff and promote staff morale to ensure residents needs are being met in a proactive 
manner. Qualifications: Must have a valid MA Nursing License. Minimum of 1 year Long term care experience/SNF 
experience preferred - although new graduates welcome. We offer great benefits for all Full Time staff -some 
limitations apply for part time staff. Child Day Care facility on site since 1986 - infants through preschoolers - 
prorated for staff. Job Type: Full-time or part-time Job Types: Full-time, Part-time Benefits: 401(k) 401(k) 
matching Dental insurance Flexible schedule Health insurance Life insurance Paid time off Referral program Tuition 
reimbursement Physical setting: JCAHO accredited facility Long term care Nursing home Standard shift: Day shift 
Evening shift Night shift Supplemental schedule: Holidays Overtime Weekly schedule: Rotating weekends COVID-19 
considerations: All staff are required to follow current COVID 19 protocols as defined by the Commonwealth of MA - 
must be prepared to wear masks, and follow other infection control protocols as well as all vaccination guidelines 
expected to be employed in a LTC SNF Ability to commute/relocate: Sudbury, MA 01776: Reliably commute or planning 
to relocate before starting work (Required) Experience: Nursing to: 1 year (Preferred) License/Certification: RN or
LPN License in Massachusetts (Required) Work Location: One location Principals only. Recruiters, please don't 
contact this job poster. Do NOT contact this job poster with unsolicited services or offers. post id: 7694393986 
updated: [ ]

In [8]:
pred = info_extract(context = test_examples[1][2])
print(pred.info.split('Answer: ')[1])

c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\transformers\models\qwen2\modeling_qwen2.py:693: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


{
    "position_title": "Nurses (RNs or LPNs) in SNF",
    "location": "Sudbury, MA 01776",
    "work_arrangement": "On-site",
    "experience": "Minimum of 1 year long term care experience, SNF experience preferred",
    "employment_type": "Full-time or Part-time",
    "pay": "Hourly",
    "degree": "RN or LPN license in Massachusetts required",
    "certifications": "RN or LPN license in Massachusetts required",
    "required_skills": "Nursing experience, ability to manage staff and promote staff morale, ability to ensure 
residents' needs are being met in a proactive manner"
}

#### Create Evaluation Set

In [9]:
test_results = test_examples_list
test_contents = list(test_examples[1])
test_examples_list = [dspy.Example(context=content, info=result) for content, result in zip(test_contents, test_results)]
testset = test_examples_list
testset = [x.with_inputs('context') for x in testset]

In [10]:
answ = testset[0]
print(answ)

Example({'context': 'Federal Sales Engineer - Tech & ISR Experience - Hybrid Remote - Walpole MA\nAdvanced 
Recruiting Solutions\nWalpole, MA\nDepends on Experience\nFull Time\nWork from home\n10% Travel\nWe are a Power 
Electronics company seeking a SENIOR \nInside Sales Rep/Sales Engineer\nwith experience in the development and 
selling of power electronics or similar hardware, along with providing technical support, to the DOD, Homeland 
Security, Prime Contractors, and Commercial Industries (Telecom, Medical, Robotics, Energy, etc.). \nMust live 
within commuting distance of Walpole, MA, USA, have sales and technical experience selling to the DOD (army 
preferred), and be willing to travel 10-20%.\nThis position will be responsible for supporting the sales team in 
the identification, qualification, and capture of power supply opportunities within government and commercial 
markets, and will work closely with the Sales and Marketing team in converting new and existing business 
opportunities. \nThe position is located in our Corporate Headquarters in Walpole, MA, and will report to the 
Senior Vice President of Sales and Marketing.\nHybrid Working Schedule.\nCompensation:\n$120K/year\nKey 
Responsibilities:\nIdentify opportunities from cold calls, web leads, reps, trade shows, meetings and industry 
networking. Support the sales team through the RFI, RFQ, capture, product qualification, LRIP (Low Rate Initial 
Production) and FRP (Full Rate Production) Phases.\nCollaborate with our engineering and operation teams in 
gathering technical information necessary in crafting a unique winning solution using our Products and Service that
meets the customer technical and business requirements.\nUsing our Products and Services to create quotes and 
proposals that include system block diagrams, compliance matrix, schedules along with other pertinent information 
necessary to respond to customer RFI, RFQs and other opportunities.\nHelp clients solve problems with installed 
equipment.\nHelp in researching, defining and coordinating evaluations of new products.\nManage account-wide sales 
forecasting and opportunity reporting.\nSupport the RMA process.\nField incoming calls and webchat 
inquiries.\nSupport sales team colleagues with the above activities.\n\nRequirements:\nA BS in the Engineering 
field\nInside/Outside Technical Sales Experience\nExperience working with Outside Sales Reps\nCRM 
(\nsalesforce.com)\nRFQ experience and price quotes to the DOD\nPreferred:\nWorking knowledge of ERP (Epicor is 
preferred)\nFamiliar with power electronics and has sales experience in the military and industrial 
sectors\nMilitary Service is considered a plus.\nWe offer a total compensation package that is competitive within 
the industry. Our compensation and benefits offerings are designed to deliver exceptional rewards to exceptional 
performers, and include paid vacation and Federal holidays, with 401(k), medical and dental insurance, individual 
performance bonus and a competitive salary (base + commission).\nReport this job\nDice Id: \n10493906\nPosition Id:
\n8008878', 'info': '{"position_title": "Senior Inside Sales Rep/Sales Engineer", "location": "Walpole, MA", 
"work_arrangement": "Hybrid", "experience": "Depends on Experience", "employment_type": "Full Time", "pay": 
"$120K/year", "degree": "A BS in the Engineering field", "certification": "CRM (salesforce.com), RFQ experience and
price quotes to the DOD", "required_skills": "Inside/Outside Technical Sales Experience, Experience working with 
Outside Sales Reps"}'}) (input_keys={'context'})

In [11]:
pred = info_extract(context = testset[0].context)
print(pred.info.split('Answer: ')[1])

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


{
    "position_title": "Federal Sales Engineer",
    "location": "Walpole, MA",
    "work_arrangement": "Hybrid",
    "experience": "5+ years",
    "employment_type": "Full Time",
    "pay": "$120k/year",
    "degree": "BS in Engineering",
    "certifications": "Salesforce.com, RFI, RFQ, CRM",
    "required_skills": "Sales and technical experience selling to the DOD, Army preferred, working knowledge of ERP
Epicor, familiar with power electronics, and sales experience in the military and industrial sectors"
}

In [12]:
validate_ans(answ,pred)

{"position_title": "senior inside sales rep/sales engineer", "location": "walpole, ma", "work_arrangement": 
"hybrid", "experience": "depends on experience", "employment_type": "full time", "pay": "$120k/year", "degree": "a 
bs in the engineering field", "certification": "crm (salesforce.com), rfq experience and price quotes to the dod", 
"required_skills": "inside/outside technical sales experience, experience working with outside sales reps"}

{
    "position_title": "federal sales engineer",
    "location": "walpole, ma",
    "work_arrangement": "hybrid",
    "experience": "5+ years",
    "employment_type": "full time",
    "pay": "$120k/year",
    "degree": "bs in engineering",
    "certifications": "salesforce.com, rfi, rfq, crm",
    "required_skills": "sales and technical experience selling to the dod, army preferred, working knowledge of erp
epicor, familiar with power electronics, and sales experience in the military and industrial sectors"
}

0.5199180327868853

0.5199180327868853

#### Run Evaluation

In [13]:
evaluation = Evaluate(devset=testset, num_threads=1, display_progress=True, display_table=10, return_outputs=True)

prev_score=evaluation(info_extract, metric=validate_ans)

  0%|          | 0/10 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


{"position_title": "senior inside sales rep/sales engineer", "location": "walpole, ma", "work_arrangement": 
"hybrid", "experience": "depends on experience", "employment_type": "full time", "pay": "$120k/year", "degree": "a 
bs in the engineering field", "certification": "crm (salesforce.com), rfq experience and price quotes to the dod", 
"required_skills": "inside/outside technical sales experience, experience working with outside sales reps"}

{
    "position_title": "federal sales engineer",
    "location": "walpole, ma",
    "work_arrangement": "hybrid",
    "experience": "5+ years",
    "employment_type": "full time",
    "pay": "$120k/year",
    "degree": "bs in engineering",
    "certifications": "salesforce.com, rfi, rfq, crm",
    "required_skills": "sales and technical experience selling to the dod, army preferred, working knowledge of erp
epicor, familiar with power electronics, and sales experience in the military and industrial sectors"
}

0.5199180327868853

...

{"position_title": "cryptocurrency / fx trader - entry level", "location": "not specified", "work_arrangement": 
"remote", "experience": "no prior experience required", "employment_type": "full-time or part-time", "pay": 
"results-based commissions and performance bonuses", "degree": "bachelor's degree in finance, economics, or related
field preferred", "certification": "none specified", "required_skills": "strong analytical skills, quick 
decision-making"}

{
    "position_title": "cryptocurrency fx trader entry level",
    "location": "remote",
    "work_arrangement": "remote",
    "experience": "no prior experience required",
    "employment_type": "contract business",
    "pay": "over $100,000 annually with the possibility of unlimited earnings",
    "degree": "bachelor's degree in finance, economics, or a related field preferred but not required",
    "certifications": "no specific certifications required",
    "required_skills": "strong motivation and drive to succeed, willingness to learn and an entrepreneurial spirit,
strong analytical skills, ability to make quick decisions in a fast-paced environment, ability to work in a 
fast-paced and mentally-challenging environment"
}

0.3989010989010989

Average Metric: 5.018342589327313 / 10  (50.2): 100%|██████████| 10/10 [04:39<00:00, 27.94s/it]


,example_context,example_info,pred_context,pred_info,validate_ans
0,"Federal Sales Engineer - Tech & ISR Experience - Hybrid Remote - Walpole MA Advanced Recruiting Solutions Walpole, MA Depends on Experience Full Time Work...","{""position_title"": ""Senior Inside Sales Rep/Sales Engineer"", ""location"": ""Walpole, MA"", ""work_arrangement"": ""Hybrid"", ""experience"": ""Depends on Experience"", ""employment_type"": ""Full Time"", ""pay"": ""$120K/year"", ""degree"": ""A BS in the...",federal sales engineer tech isr experience hybrid remote walpole ma advanced recruiting solutions walpole ma depends on experience full time work from home 10 travel...,"key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words --- Context: federal sales engineer tech isr...",✔️ [0.5199180327868853]
1,"Position Description Penetration Tester Location Washington, DC Req # 12763 # of openings 2 ECS is seeking a Penetration Tester to work in our Washington,...","{""position_title"": ""Penetration Tester"", ""location"": ""Washington, DC"", ""work_arrangement"": ""On-site"", ""experience"": ""10+ years of Penetration Testing experience"", ""employment_type"": ""Full time"", ""pay"": ""Not specified"", ""degree"": ""Bachelors Degree in...",position description penetration tester location washington dc req 12763 of openings 2 ecs is seeking a penetration tester to work in our washington dc office...,"key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words --- Context: position description penetration tester location...",✔️ [0.22915032679738562]
2,NURSES - RNS & LPNs IN SNF - SIGN ON BONUS - CHILD DAYCARE ON SITE (Sudbury) Sudbury Pines Extended Care Facility Inc. compensation: HOURLY...,"{""position_title"": ""NURSES - RNs or LPNs"", ""location"": ""Sudbury, MA 01776"", ""work_arrangement"": ""on-site"", ""experience"": ""Minimum of 1 year Long term care experience/SNF experience preferred"", ""employment_type"": ""full-time"",...",nurses rns lpns in snf sign on bonus child daycare on site sudbury sudbury pines extended care facility inc compensation hourly every other weekend required...,"key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words --- Context: nurses rns lpns in snf...",✔️ [0.5577922077922077]
3,Planner IV - Transportation Planner Job Details Apply Print Share This listing closes on 7/17/2023 at 11:59 PM Pacific Time (US & Canada); Tijuana. Salary...,"{""position_title"": ""Planner IV - Transportation Planner"", ""location"": ""Yakima, WA, 98901"", ""work_arrangement"": ""On-site"", ""experience"": ""5 years of increasingly responsible professional experience"", ""employment_type"": ""Full-Time"", ""pay"": ""$39.84 -...",planner iv transportation planner job details apply print share this listing closes on 7172023 at 1159 pm pacific time us canada tijuana salary 3984 5053...,"key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words --- Context: planner iv transportation planner job...",✔️ [0.6839727195225916]
4,"Associate Attorney Juan Ramos Law Group, PLLC McAllen, TX Job Details Full-time From $50,000 a year 1 day ago Qualifications Spanish Law Doctoral degree Criminal...","{""position_title"": ""Associate Attorney"", ""location"": ""McAllen, TX"", ""work_arrangement"": ""on-site"", ""experience"": ""None specified."", ""employment_type"": ""full-time"", ""pay"": ""$50,000 a year"", ""degree"": ""Law doctoral degree"", ""certification"": ""Admission to the...",associate attorney juan ramos law group pllc mcallen tx job details fulltime from 50000 a year 1 day ago qualifications spanish law doctoral degree

#### Log Evaluation to mlFlow

In [14]:
context = [example[0].context for example in prev_score[1]]
gold_example = [example[0].info for example in prev_score[1]]
prediction = [example[1].info.split("Answer: ")[1].replace('\n', '') for example in prev_score[1]]
score = [example[2] for example in prev_score[1]]
df = pd.DataFrame({
    'context': context,
    'gold': gold_example,
    'prediction': prediction,
    'score': score
})

In [15]:
mlflow.log_table(data=df, artifact_file="qa_info_eval.json")
mlflow.log_metric('overall_rouge_score', prev_score[0])

In [16]:
mlflow.end_run()